# Agregação dos Dados 🎒 🎲

<pre>A base de indicadores criminais por município e ano do Estado do Rio Grande do Sul é publicada no formato Excel. Por meio de raspagem de dados, obtivemos os dados referentes aos anos de 2012 até a presente data.</pre>

<pre>Execute as duas células abaixo para condensar todos os anos da base de indicadores criminais por município do Estado do Rio Grande do Sul em um único DataFrame, de nome <b>df_crimes</b>.</pre>

In [ ]:
!pip install --upgrade openpyxl --quiet
!pip install --upgrade pandas --quiet

In [ ]:
import pandas as pd

dfs = []

xlsx = pd.ExcelFile('data/crimes_rs_municipios.xlsx')

anos = [year for year in xlsx.sheet_names if all(c.isdigit() for c in year)]

for ano in anos:
    df = pd.read_excel(xlsx, sheet_name=ano)
    df['Ano'] = ano
    dfs.append(df)

df_crimes = pd.concat(dfs)
df_crimes = df_crimes.drop(columns='Total/Ano')
df_crimes.head()

<pre>Crie um DataFrame, chamado <b>df_contagem</b>, contendo o nome do município e a contagem de anos presentes no DataFrame <b>df_crimes</b>.</pre>

👉 dica: utilize o método `groupby` do DataFrame `df_crimes`.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_contagem = df_crimes[['ano','Municípios']].groupby('Municípios').count()
```

<br/>

</details>

<pre>Utilize, em seguida, o método `describe` para verificar se há dados para todos os municípios, referente ao período de 2012-2024.</pre>

In [ ]:
df_contagem.describe()

<pre>Liste, se existirem, os municípios que possuem menos de 13 registros presentes na base de dados.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_contagem[df_contagem['Ano'] < 13]
```

<br/>

</details>

🚨 Observe que todos os municípios têm os 13 registros presentes na base de dados. 🚨

<pre>Agora, crie um DataFrame, chamado <b>df_crimes_municipios</b>, contendo o nome do município e a soma de cada crime presente no DataFrame <b>df_crimes</b>. Não inclua a coluna <b>Ano</b> no resultado.</pre>

👉 dica: utilize o método `groupby` do DataFrame `df_crimes`, bem como o método `drop`.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_crimes_municipios = df_crimes.drop(columns='Ano').groupby('Municípios').sum()
```

<br/>

</details>

<pre>Agora, vamos verificar se o comportamento observado na quantidade de crimes de estelionato, furto e roubo de veículos em Porto Alegre se repete em todo o Estado do Rio Grande do Sul.</pre>

<pre>Primeiro, com indexação booleana, mantenha apenas o município de <b>PORTO ALEGRE</b> e atribua o DataFrame resultante à variável <b>df_crimes_poa</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_crimes_poa = df_crimes[df_crimes['Municípios'] == 'PORTO ALEGRE']
```

<br/>

</details>

<pre>Agora, remova a coluna <b>Municípios</b> e coloque a coluna <b>Ano</b> como índice do DataFrame <b>df_crimes_poa</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_crimes_poa = df_crimes_poa.drop(columns='Municípios').set_index('Ano')
```

<br/>

</details>

<pre>Por fim, execute a célula abaixo para criar um gráfico de linha com as quantidades anuais dos crimes de estelionato, furto e roubo de veículos no município de Porto Alegre.</pre>

In [ ]:
df_crimes_poa[['Estelionato','Furto de Veículo','Roubo de Veículo']].plot();

<pre>Agora, repita os passos anteriores, mas removendo (e não apenas mantendo) o município de <b>PORTO ALEGRE</b> e atribua o DataFrame resultante à variável <b>df_crimes_rs</b>.</pre>

👉 lembre-se de usar indexação booleana para remover a cidade de `PORTO ALEGRE` e agrupar o DataFrame resultante por `Ano`, aplicando o método `sum`.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_crimes_rs = df_crimes[df_crimes['Municípios'] != 'PORTO ALEGRE']
df_crimes_rs = df_crimes_rs.groupby('Ano').sum()
```

<br/>

</details>

<pre>Por fim, execute a célula abaixo para criar um gráfico de linha com as quantidades anuais dos crimes de estelionato, furto e roubo de veículos no Estado do Rio Grande do Sul, sem contar com o município de Porto Alegre.</pre>

In [ ]:
df_crimes_rs[['Estelionato','Furto de Veículo','Roubo de Veículo']].plot();

<pre>Em vez de comparar municípios, vamos avaliar os índices de criminalidade por unidades geográficas maiores, como as regiões geográficas intermediárias e imediatas. Para isso, vamos fazer a junção do DataFrame <b>df_crimes</b> com uma tabela de definição de regiões geográficas intermediárias e imediatas e agrupar os dados por elas.</pre>

🚨 <a href="https://biblioteca.ibge.gov.br/visualizacao/livros/liv100600.pdf">As regiões geográficas intermediárias e imediatas</a> foram definidas pelo IBGE com o objetivo de agrupar municípios vizinhos que possuam uma integração socioeconômica e funcional mais próxima, facilitando assim a análise de dados e a compreensão da dinâmica regional. 🚨

<pre>O IBGE disponibiliza os dados de municípios, suas microrregiões e egiões geográficas intermediárias e imediatas do Rio Grande do Sul por meio de uma <a href="https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios">API</a>. Execute a célula abaixo para obter os dados e amarzená-los no DataFrame <b>df_rs</b>.</pre>

In [ ]:
import unicodedata

df_rs = pd.read_json('https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios')
df_rs['regiao-imediata'] = df_rs['regiao-imediata'].apply(lambda coluna: coluna['nome'])
df_rs['microrregiao'] = df_rs['microrregiao'].apply(lambda coluna: coluna['nome'])
df_rs['nome'] = df_rs['nome'].apply(lambda coluna: unicodedata.normalize('NFKD', coluna).encode('ASCII', 'ignore').decode('UTF-8').upper())
df_rs = df_rs.drop(columns='id')

<pre>Agora, faça a mescla do DataFrame de crimes (<b>df_crimes</b>) com o DataFrame de regiões geográficas intermediárias e imediatas (<b>df_rs</b>), usando as colunas <b>Municípios</b> e <b>nome</b>, respectivamente, e atribua o DataFrame resultante à variável <b>df_crimes_regioes</b>.</pre>

👉 Lembre-se de remover a coluna `nome` depois da operação de mescla, por se tratar de dado repetido.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_crimes_regioes = pd.merge(left=df_crimes, right=df_rs, left_on='Municípios', right_on='nome').drop(columns=['nome'])
```

<br/>

</details>

<pre>Agora, atualize o DataFrame da variável <b>df_crimes_regioes</b>, contendo o nome das regiões geográficas intermediárias e imediatas e a soma de cada crime presente no DataFrame. Não inclua a coluna <b>Ano</b> no resultado, nem qualquer outra coluna não numérica.</pre>

👉 dica: utilize o método `groupby` do DataFrame `df_crimes` e o método `sum` com o atributo `numeric_only` verdadeiro.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_crimes_regioes = df_crimes_regioes.drop(columns='Ano').groupby('regiao-imediata').sum(numeric_only=True)
```

<br/>

</details>

<pre>Por fim, liste as 10 regiões geográficas intermediárias e imediatas com o maior número de homicídios dolosos.</pre>

👉 dica: use o método `nlargest` do DataFrame `df_crime_regioes`.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_crimes_regioes.nlargest(10, 'Homicídio Doloso')
```

<br/>

</details>

<pre>Agora, execute a célula abaixo para criar um gráfico de pizza com as quantidades anuais dos crimes de homicídio doloso nas 10 regiões geográficas intermediárias e imediatas com mais ocorrências desse crime no Estado do Rio Grande do Sul.</pre>

In [ ]:
df_crimes_regioes.nlargest(10, 'Homicídio Doloso').plot(kind='pie', y='Homicídio Doloso', legend=False);

<pre>Agora, calcularemos a taxa de homicídio doloso por 100.000 habitantes nas regiões geográficas intermediárias e imediatas.</pre>

<pre>Execute a célula abaixo para obter a população estimada de cada município do Rio Grande do Sul, conforme último ano estimado pelo IBGE, e armazenar o resultado no DataFrame atribuído à variável <b>df_pop_rs</b>.</pre>

In [24]:
df_pop_rs = pd.read_csv(r'https://sidra.ibge.gov.br/geratabela?format=br.csv&name=tabela6579.csv&terr=NC&rank=-&query=t/6579/n6/all/v/all/p/last%201/l/v,p,t', skiprows=3, nrows=5570, sep=';', dtype={'Cód.': 'str'})
df_pop_rs = df_pop_rs[df_pop_rs['Cód.'].str.startswith('43')].drop(columns='Cód.').rename(columns={df_pop_rs.columns[-1]: 'Pop. Estimada'})
df_pop_rs['Município'] = df_pop_rs['Município'].str[:-5].map(lambda coluna: unicodedata.normalize('NFKD', coluna).encode('ASCII', 'ignore').decode('UTF-8').upper())
df_pop_rs.to_json('data/pop_rs.json')

<pre>Agora, faça a mescla do DataFrame de regiões geográficas intermediárias e imediatas (<b>df_rs</b>) com o DataFrame de população estimada dos municípios do Rio Grande do Sul (<b>df_pop_rs</b>), usando as colunas <b>nome</b> e <b>Município</b>, respectivamente, e atribua o DataFrame resultante à variável <b>df_pop_regioes</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_pop_regioes = pd.merge(left = df_rs, right = df_pop_rs, left_on = 'nome', right_on = 'Município')
```

<br/>

</details>

<pre>Por fim, atualize o DataFrame armazenado na variável <b>df_pop_regioes</b> com a população estimada das regiões geográficas intermediárias e imediatas.</pre>

👉 dica: use o método `groupby` e o método `sum` com parâmetro `numeric_only` verdadeiro.

<details>
  <summary>Resposta</summary>

<br/>

```python
df_pop_regioes = df_pop_regioes.groupby('regiao-imediata').sum(numeric_only=True)
```

<br/>

</details>

<pre>Por fim, utilize os dados populacionais para calcular a taxa de homicídio doloso por 100.000 habitantes por região geográfica intermediária e imediata, atribuindo o resultado à coluna <b>Taxa Homicídio Doloso</b> e o DataFrame resultante à variável <b>df_taxa_homicidio</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_taxa_homicidio = df_pop_regioes.join(df_crimes_regioes)[['Homicídio Doloso', 'Pop. Estimada']]
df_taxa_homicidio['Taxa Homicídio Doloso'] = 100_000 * df_taxa_homicidio['Homicídio Doloso'] / df_taxa_homicidio['Pop. Estimada']
```

<br/>

</details>

<pre>Agora, execute a célula abaixo para criar um gráfico de barras horizontal com taxa de homicídio doloso por 100.000 habitantes nas 10 regiões geográficas intermediárias e imediatas que lideram o ranking dessa taxa no Estado do Rio Grande do Sul.</pre>

In [ ]:
df_taxa_homicidio.nlargest(10, 'Taxa Homicídio Doloso').plot.barh(y='Taxa Homicídio Doloso', title='Top 10 de taxa de homicídio doloso por 100.000 habitantes por região geográfica intermediária e imediata', legend=False);

## Explorando preços de combustíveis

<pre>Execute a célula abaixo para carregar o DataFrame com o resultado da última pesquisa semanal referente ao Levantamento de Preços de Combustíveis, segregados por abrangência geográfica, disponibilizado pela <a href="https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/levantamento-de-precos-de-combustiveis-ultimas-semanas-pesquisadas">Agência Nacional do Petróleo, Gás Natural e Biocombustíveis</a>.</pre>

In [25]:
from bs4 import BeautifulSoup
import requests

url = 'https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/levantamento-de-precos-de-combustiveis-ultimas-semanas-pesquisadas'
html = requests.get(url).text
hrefs = [a_tag['href'] for a_tag in BeautifulSoup(html, 'html.parser').find_all('a') if all(href_substrings in a_tag['href'].lower() for href_substrings in ['revendas_lpc', '.xlsx'])]
df_precos_recentes = pd.read_excel(hrefs[0])
df_precos_recentes = df_precos_recentes.dropna(thresh=df_precos_recentes.shape[1]-2)
df_precos_recentes.columns = df_precos_recentes.iloc[0]
df_precos_recentes = df_precos_recentes.iloc[1:]
df_precos_recentes.to_json('data/precos_combustiveis_ultima_semana.json')

<pre>Agora, vamos responder à bateria de perguntas abaixo sobre preço de combustíveis.</pre>

<pre>Conte quantos municípios por Unidade da Federação estão presentes no DataFrame <b>df_precos_recentes</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_precos_recentes[['ESTADO', 'MUNICÍPIO']].drop_duplicates().groupby('ESTADO').count()
```

<br/>

</details>

<pre>Calcule os preços mínimo, médio e máximo dos preços de revenda de gasolina comum por Unidade da Federação.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_precos_descritivos_recentes = df_precos_recentes[df_precos_recentes['PRODUTO'] == 'GASOLINA COMUM'].groupby('ESTADO').agg({'PREÇO DE REVENDA': ['min', 'mean', 'max']}).astype('float').droplevel(0, 'columns')
```

<br/>

</details>

In [ ]:
df_precos_descritivos_recentes = df_precos_recentes[df_precos_recentes['PRODUTO'] == 'GASOLINA COMUM'].groupby('ESTADO').agg({'PREÇO DE REVENDA': ['min', 'mean', 'max']}).astype('float')

<pre>Agora, responda:
 - Qual Unidade da Federação apresenta o menor preço de revenda de gasolina comum?
 - Qual Unidade da Federação apresenta o maior preço de revenda de gasolina comum?</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_precos_descritivos_recentes.nsmallest(1, ('PREÇO DE REVENDA', 'min'))
```

<br/>

</details>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_precos_descritivos_recentes.nlargest(1, ('PREÇO DE REVENDA', 'max'))
```

<br/>

</details>

<pre>Por fim, informe qual o menor, médio e maior preços de revenda de gasolina comum por bairro do Distrito Federal.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df_precos_recentes[(df_precos_recentes['ESTADO'] == 'DISTRITO FEDERAL') & (df_precos_recentes['PRODUTO'] == 'GASOLINA COMUM')].groupby('BAIRRO').agg({'PREÇO DE REVENDA': ['min', 'mean', 'max']})
```

<br/>

</details>